## 3. Modeling TF-IDF x LSTM

### 3.1 Importing the Libraries 

In [21]:
import pickle
import os 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input, Concatenate # <-- Hanya import LSTM
from tensorflow.keras.callbacks import EarlyStopping

In [22]:
MODEL_SAVE_PATH = "models/lstm_model.keras"

### 3.2 Define the Parameter

In [23]:
VOCAB_SIZE = 5000
TFIDF_FEATURES = 5000
MAX_LEN = 200

### 3.2 Load the Dataset

In [24]:
X_train_pad   = pickle.load(open("../dataset/processed/X_train_pad.pkl", "rb"))
X_train_tfidf = pickle.load(open("../dataset/processed/X_train_tfidf.pkl", "rb"))
y_train       = pickle.load(open("../dataset/processed/y_train.pkl", "rb"))

X_test_pad    = pickle.load(open("../dataset/processed/X_test_pad.pkl", "rb"))
X_test_tfidf  = pickle.load(open("../dataset/processed/X_test_tfidf.pkl", "rb"))
y_test        = pickle.load(open("../dataset/processed/y_test.pkl", "rb"))

### 3.3 Check Array Dimension

In [25]:
print(f"Shape X_train_pad: {X_train_pad.shape}")
print(f"Shape X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")

Shape X_train_pad: (2526, 200)
Shape X_train_tfidf: (2526, 5000)
Shape y_train: (2526,)
Shape y_test: (632,)


### 3.4 Build LSTM Model with TF-IDF

In [26]:
# input squence for lstm
input_seq = Input(shape=(MAX_LEN,), name='input_sequence')
x = Embedding(input_dim=VOCAB_SIZE, output_dim=128)(input_seq)

x = LSTM(64)(x) 

# vektor tf-idf
input_tfidf = Input(shape=(TFIDF_FEATURES,), name='input_tfidf')
y = Dense(32, activation='relu')(input_tfidf)

# concat
combined = Concatenate()([x, y])

# layer
z = Dense(64, activation="relu")(combined)
output_layer = Dense(1, activation="sigmoid")(z)

# define model
model = Model(inputs=[input_seq, input_tfidf], outputs=output_layer)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
print(model.summary())

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 200, 128)  │    640,000 │ input_sequence[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_tfidf         │ (None, 5000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     49,408 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │    160,032 │ input_tfidf[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 96)        │          0 │ lstm_2[0][0],     │
│ (Concatenate)       │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      6,208 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         65 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 855,713 (3.26 MB)

 Trainable params: 855,713 (3.26 MB)

 Non-trainable params: 0 (0.00 B)

None


### 3.5 Define Callbacks

In [27]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True,
    verbose=1
)

### 3.6 Training

In [28]:
history = model.fit(
    [X_train_pad, X_train_tfidf],
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 133ms/step - accuracy: 0.8446 - loss: 0.5085 - val_accuracy: 0.9387 - val_loss: 0.1818
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 129ms/step - accuracy: 0.9678 - loss: 0.0943 - val_accuracy: 0.9862 - val_loss: 0.0700
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - accuracy: 0.9946 - loss: 0.0208 - val_accuracy: 0.9743 - val_loss: 0.0964
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - accuracy: 0.9995 - loss: 0.0062 - val_accuracy: 0.9763 - val_loss: 0.0878
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 124ms/step - accuracy: 0.9975 - loss: 0.0056 - val_accuracy: 0.9585 - val_loss: 0.1311
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


### 3.7 Evaluation

In [29]:
loss, acc = model.evaluate([X_test_pad, X_test_tfidf], y_test, verbose=1)
print(f"\nAkurasi Test (LSTM + TF-IDF): {acc:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9668 - loss: 0.0824

Akurasi Test (LSTM + TF-IDF): 0.9668


### 3.8 Save Model

In [30]:
os.makedirs("models", exist_ok=True)
model.save(MODEL_SAVE_PATH)
print("Model LSTM saved to:", MODEL_SAVE_PATH)

Model LSTM saved to: models/lstm_model.keras
